### Dependencies

In [1]:
from apify_client import ApifyClient
from dotenv import load_dotenv
import pandas as pd
import re
import requests
import json
import os
import shutil
from datetime import datetime
from pathlib import Path
from apify_class import Apify

In [2]:
import importlib
import apify_class

importlib.reload(apify_class)

<module 'apify_class' from 'c:\\Users\\ismai\\OneDrive\\Desktop\\UpClout\\src\\apify_class.py'>

In [3]:
DATA_PATH = "../data"

In [3]:
def clean_post_data(current_folder="../data/mahirahkhan") -> list[dict]:

    json_file = None
    for file in os.listdir(current_folder):
        if file.endswith(".json"):
            json_file = os.path.join(current_folder, file)
            break

    with open(json_file, "r", encoding="utf-8") as file:
        data=json.load(file)
   
    return data

dict_list = clean_post_data()

In [9]:
_dict = dict_list[41]

In [10]:
_dict

{'id': '3635506738968734308',
 'type': 'Sidecar',
 'shortCode': 'DJz6h4KxSpk',
 'caption': '\U0001fa75\n\nOutfit: @iqbalhussainofficial \nStyling: @sananver\nEarrings & ring : @amnashariff.jewelry \nBangles : @traditionaljewelryy\nShoes: @balenciaga\nHair & makeup: @mubsher.bhatti \nImages: @rehanmithanii',
 'hashtags': [],
 'mentions': ['iqbalhussainofficial',
  'sananver',
  'amnashariff.jewelry',
  'traditionaljewelryy',
  'balenciaga',
  'mubsher.bhatti',
  'rehanmithanii'],
 'url': 'https://www.instagram.com/p/DJz6h4KxSpk/',
 'commentsCount': 527,
 'firstComment': 'Gorgeous 🖤',
 'latestComments': [{'id': '18055474091386690',
   'text': 'Gorgeous 🖤',
   'ownerUsername': 'i_am_opurbo',
   'ownerProfilePicUrl': 'https://scontent-atl3-3.cdninstagram.com/v/t51.2885-19/538942557_18076719293050954_1703258559465848059_n.jpg?stp=dst-jpg_s150x150_tt6&efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_nc_ht=scontent-atl3-3.cdninstagram.com&_nc_cat=110&_nc_oc=Q6cZ2QG141oZ-yEGj_c

In [11]:
keys_of_interest = [
    "id",
    "type",
    "caption",
    "hashtags", #list
    "mentions", #list
    "url",
    "commentsCount",
    "likesCount",
    "timestamp",
    "ownerUsername",
    "ownerId",
    "taggedUsers", # dict within a list
    "coauthorProducers", # dict within a list
    "ownerFullName",
    "isSponsored",
]

In [12]:
new_dict = {k: v for k, v in _dict.items() if k in keys_of_interest}
new_dict

{'id': '3635506738968734308',
 'type': 'Sidecar',
 'caption': '\U0001fa75\n\nOutfit: @iqbalhussainofficial \nStyling: @sananver\nEarrings & ring : @amnashariff.jewelry \nBangles : @traditionaljewelryy\nShoes: @balenciaga\nHair & makeup: @mubsher.bhatti \nImages: @rehanmithanii',
 'hashtags': [],
 'mentions': ['iqbalhussainofficial',
  'sananver',
  'amnashariff.jewelry',
  'traditionaljewelryy',
  'balenciaga',
  'mubsher.bhatti',
  'rehanmithanii'],
 'url': 'https://www.instagram.com/p/DJz6h4KxSpk/',
 'commentsCount': 527,
 'likesCount': 74734,
 'timestamp': '2025-05-18T22:09:57.000Z',
 'ownerFullName': 'Mahira Khan',
 'ownerUsername': 'mahirahkhan',
 'ownerId': '666588648',
 'taggedUsers': [{'full_name': 'IQBAL HUSSAIN',
   'id': '1512515216',
   'is_verified': True,
   'profile_pic_url': 'https://scontent-atl3-3.cdninstagram.com/v/t51.2885-19/305816293_816761502988870_2731701604349564583_n.jpg?stp=dst-jpg_s150x150_tt6&efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_

In [ ]:
def handle_taggedUsers(postID: int, taggedUsers: list):
    for user_dict in taggedUsers:
        id = int(user_dict.get("id"))
        username = user_dict.get("username")
        # Storing in Table        
        #postgres.load_taggedUsers(id, username)


In [42]:
user_dict = handle_taggedUsers(_dict.get("id"), _dict.get("taggedUsers"))
user_dict

1512515216 iqbalhussainofficial


In [61]:
from datetime import datetime, timezone

def correct_dtypes_post(_dict: dict) -> dict:
    convert_to_int = ['id', 'commentsCount', 'likesCount', 'ownerId']

    for key, value in _dict.items():
        if key == 'timestamp' and isinstance(value, str):
            try:
                _dict[key] = datetime.strptime(value, "%Y-%m-%dT%H:%M:%S.%fZ").replace(tzinfo=timezone.utc)
            except ValueError:
                _dict[key] = datetime.strptime(value, "%Y-%m-%dT%H:%M:%S%z").astimezone(timezone.utc)

        elif key in convert_to_int:
            try:
                _dict[key] = int(value)
            except (ValueError, TypeError):
                _dict[key] = None

        elif isinstance(value, (list, dict)) or value is None:
            # keep lists, dicts, and None as they are
            _dict[key] = value

        else:
            _dict[key] = str(value)

    return _dict

In [ ]:
from load import Postgres
import psycopg2

In [155]:
def potential_influencers(username: str):
    # scrape meta deta of username
    # check if following is greater than threshold (1000) 
    # append username to txt file
    # else delete scraped data
    apify = Apify()
    response=apify.scrape_meta_data(username)

    print(response)

    if response == 0:
        print("Already in Database")
        return

    folder_path = f"../data/{username}"
    df=pd.read_csv(f"{folder_path}/{username}_meta_data.csv")

    threshold: int = 1000
    followers = int(df['followersCount'][0])
    if followers > threshold:
        print("Potential Influencer")
    else:
        # delete folder
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
            print(f"Folder {folder_path} deleted successfully!")

In [ ]:
def check_mentions(new_dict: dict) -> None:
    mentions = new_dict['mentions']
    if not mentions:
        return
    
    for username in mentions:
        # TODO: send username to a function that determines if it is indeed an influencer worth keeping in database 
        #potential_influencers(username)
        ...

In [162]:
potential_influencers("humzaamin")

Folder for humzaamin already exists. Skipping...
0
Already in Database


In [ ]:
check_mentions(new_dict)

In [163]:
new_dict['mentions']

['emaandharani',
 'emaandharanistyled',
 'iambabarzaheer',
 'mubsher.bhatti',
 'shahbazshaziofficial']

In [164]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url1="https://starngage.com/plus/en/brand/ranking/instagram/pakistan/politics"

url="https://starngage.com/plus/en/influencer/ranking/instagram/pakistan"
driver = webdriver.Chrome()  # or webdriver.Firefox()
driver.get(url)

# Wait for the table to load
wait = WebDriverWait(driver, 10)
tbody = wait.until(EC.presence_of_element_located((By.TAG_NAME, "tbody")))

# Find all name links
name_links = driver.find_elements(By.CSS_SELECTOR, "tbody tr .name a")
names = [link.text for link in name_links if link.text.strip()]

driver.quit()

cleaned_names = [name.lstrip('@') for name in names]

print(len(cleaned_names))

with open("../insta_profiles.txt", "a") as file:
    for username in cleaned_names:
        file.write(username + "\n")

100


In [140]:
with open("../data/brand_data.json", "r", encoding="utf-8") as file:
    data=json.load(file)

top_posts=data[1]['topPosts']

In [141]:
top_posts[29]['mentions']

[]

In [142]:
usernames=[]
mentions=[]

for index_2 in range(29):
    try:
        mentions.append(top_posts[index_2]['mentions'])
    except IndexError as e:
        print(f"Stopped at inner length: {index_2}\nError: {e}")

In [107]:
usernames=set(usernames)

In [143]:
mentions=[lst for lst in mentions if lst]
# Flatten the list
mentions = [username for sublist in mentions for username in sublist]

In [129]:
len(usernames)

0

In [144]:
len(mentions)

7

In [145]:
mentions

['nishatemporium',
 'Winter',
 'oriflamewithnazish',
 'am_brandstore',
 '03043888895',
 'khizan_official1',
 'khizanbts']

In [104]:
with open("../brand_profiles.txt", "a") as file:
    for username in usernames:
        file.write(username + "\n")
